# Multi-Head Latent Attention (MLA)（多头潜在注意力）

**难度：** Hard

**函数签名：** `mla_attention(X, W_dkv, W_uk, W_uv, W_q, num_heads) -> Tensor`

**参数：**
- `X` — 输入张量 (B, S, D)
- `W_dkv` — KV 压缩矩阵 (D, kv_rank)
- `W_uk` — K 解压矩阵 (kv_rank, num_heads * D_h)
- `W_uv` — V 解压矩阵 (kv_rank, num_heads * D_h)
- `W_q` — Q 投影矩阵 (D, num_heads * D_h)
- `num_heads` — 注意力头数

**返回：** 输出张量 (B, S, num_heads * D_h)

**核心思想：** 不缓存完整 K/V，而是压缩为低秩潜在向量 c_kv，推理时即时解压

**提示：** 压缩 → 解压 → 分头 → 缩放点积注意力 → 合并输出

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def mla_attention(X, W_dkv, W_uk, W_uv, W_q, num_heads):
    B, S, D = X.shape
    # D_h: 每个注意力头的维度
    # 从 W_q 的输出维度反推: W_q 形状 (D, num_heads * D_h)
    D_h = W_q.shape[1] // num_heads

    # ---- Step 1: KV 压缩 — MLA 的核心创新 ----
    # 普通 MHA: K = X @ W_k, V = X @ W_v，各自 (B, S, num_heads*D_h)
    # MLA: 先压缩到低秩空间 c_kv，大幅减少 KV 缓存
    # c_kv 形状: (B, S, kv_rank)，kv_rank << num_heads*D_h
    # 推理时只需缓存 c_kv，而非完整的 K 和 V
    c_kv = X @ W_dkv

    # ---- Step 2: 从压缩向量解压 K 和 V ----
    # K = c_kv @ W_uk，V = c_kv @ W_uv
    # 形状: (B, S, num_heads * D_h)
    # 这相当于两步矩阵乘法: X @ W_dkv @ W_uk = X @ (W_dkv @ W_uk)
    # 但拆成两步可以在推理时只缓存 c_kv
    K = c_kv @ W_uk
    V = c_kv @ W_uv

    # ---- Step 3: 投影查询 ----
    # Q = X @ W_q，形状: (B, S, num_heads * D_h)
    # Q 不需要压缩，因为 Q 不会被缓存
    Q = X @ W_q

    # ---- Step 4: 拆分多头 ----
    # 将最后一维 (num_heads * D_h) 拆成 (num_heads, D_h)
    # 然后 transpose 把 num_heads 维放到第1维
    # 最终: (B, num_heads, S, D_h)
    def split_heads(t):
        return t.view(B, S, num_heads, D_h).transpose(1, 2)

    Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

    # ---- Step 5: 缩放点积注意力 ----
    # Q @ K^T: (B, H, S, D_h) @ (B, H, D_h, S) → (B, H, S, S)
    # scale = 1/√D_h 防止点积过大
    scale = D_h ** -0.5
    attn = torch.softmax(Q @ K.transpose(-2, -1) * scale, dim=-1)

    # ---- Step 6: 加权求和 + 合并多头 ----
    # attn @ V: (B, H, S, S) @ (B, H, S, D_h) → (B, H, S, D_h)
    # transpose + reshape: (B, S, num_heads * D_h) = (B, S, D)
    out = (attn @ V).transpose(1, 2).reshape(B, S, num_heads * D_h)
    return out

In [ ]:
torch.manual_seed(0)
B, S, D = 2, 6, 32
num_heads = 4
D_h = 8          # head dim
D_c = 8          # compressed KV dim (latent)

W_dkv = torch.randn(D, D_c) * 0.1      # compress to latent
W_uk  = torch.randn(D_c, num_heads * D_h) * 0.1  # up-project to K
W_uv  = torch.randn(D_c, num_heads * D_h) * 0.1  # up-project to V
W_q   = torch.randn(D, num_heads * D_h) * 0.1

X = torch.randn(B, S, D)

# Show compression: c_kv is much smaller than full KV
c_kv = X @ W_dkv
K_full = c_kv @ W_uk
print(f"Input shape:          {X.shape}")       # (2, 6, 32)
print(f"Compressed KV shape:  {c_kv.shape}")    # (2, 6, 8)  <-- small latent
print(f"Full K shape:         {K_full.shape}")  # (2, 6, 32) <-- expanded

out = mla_attention(X, W_dkv, W_uk, W_uv, W_q, num_heads)
print(f"Output shape:         {out.shape}")     # (2, 6, 32)

In [ ]:
from torch_judge import check
check("mla")

## 📝 核心思路总结

1. **MLA 核心**：KV 先压缩到低秩潜在空间 c_kv，推理时再解压，大幅节省显存
2. **压缩-解压**：`c_kv = X @ W_dkv`（压缩），`K = c_kv @ W_uk`（解压）
3. **Q 不压缩**：因为 Q 不需要缓存，只有 K/V 需要
4. **等价性**：`X @ W_dkv @ W_uk` 等价于 `X @ W_k`，但拆分后推理更高效